# Local Backend and API Contracts

This notebook proves that the repository-local Python package and native CPU
library are used. It then checks the native ABI, deterministic execution,
finite nonzero output, CPML acquisition warnings, and invalid-input guards.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
candidates = [cwd, cwd / "tests"]
candidates.extend(parent / "tests" for parent in cwd.parents)
NOTEBOOK_DIR = next(
    (path for path in candidates if (path / "verification_utils.py").is_file()),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError("verification_utils.py was not found from the current directory.")
notebook_path = str(NOTEBOOK_DIR)
if notebook_path not in sys.path:
    sys.path.insert(0, notebook_path)

import verification_utils as vu

REPO_ROOT = vu.configure_local_import()
for module_name in tuple(sys.modules):
    if module_name == "DeepGPR" or module_name.startswith("DeepGPR."):
        del sys.modules[module_name]
import DeepGPR

LOADED_PACKAGE = vu.assert_local_deepgpr(DeepGPR, REPO_ROOT)
print(f"Repository root: {REPO_ROOT}")
print(f"DeepGPR package: {LOADED_PACKAGE}")


In [ ]:
import warnings
import torch

torch.manual_seed(2026)
DEVICE = torch.device("cpu")
CHECKS = []
METADATA = vu.runtime_metadata(DeepGPR, DEVICE)
print(METADATA)

library = DeepGPR.get_deepgpr_lib(DEVICE)
required_symbols = ("forward", "backward", "set_fdtd_order", "deepgpr_abi_version")
missing_symbols = [name for name in required_symbols if not hasattr(library, name)]
vu.record_check(
    CHECKS,
    "native ABI and required exports",
    METADATA["native_abi"] == 4 and not missing_symbols,
    abi=METADATA["native_abi"],
    missing_symbols=missing_symbols,
    library=METADATA["native_library"],
)

required_wavelets = (
    "ricker",
    "gaussian",
    "gaussian_derivative",
    "morlet",
    "sine_burst",
)
missing_wavelets = [
    name for name in required_wavelets
    if not callable(getattr(DeepGPR.wavelet, name, None))
]
vu.record_check(
    CHECKS,
    "wavelet module exports and legacy Ricker alias",
    not missing_wavelets
    and DeepGPR.ricker is DeepGPR.wavelet.ricker,
    missing_wavelets=missing_wavelets,
    legacy_alias_matches=DeepGPR.ricker is DeepGPR.wavelet.ricker,
)


In [ ]:
nx, ny, nt = 20, 24, 160
dx, dt, pml = 0.02, 3.0e-11, 4
er = torch.full((nx, ny), 4.0)
se = torch.full_like(er, 2.0e-4)
source = DeepGPR.wavelet.ricker(3.0e8, nt, dt, 1.0 / 3.0e8).reshape(1, nt, 1)
source_location = torch.tensor([[[6, 8, 0]]], dtype=torch.int32)
receiver_location = torch.tensor(
    [[[6, 12, 0], [6, 16, 0]]], dtype=torch.int32
)

def smoke_run():
    return DeepGPR.compute(
        device=DEVICE,
        dx=dx,
        dt=dt,
        source_amplitudes=source,
        source_location=source_location,
        receiver_location=receiver_location,
        er=er,
        se=se,
        pmlthick=pml,
        fdtd_order=2,
        mode=2,
        debug=True,
    )

result_a = smoke_run()
result_b = smoke_run()
receiver_a, receiver_b = result_a[-1], result_b[-1]
vu.assert_finite("smoke receiver", receiver_a, receiver_b)
vu.record_check(
    CHECKS,
    "finite nonzero CPU smoke response",
    float(receiver_a.abs().max()) > 0.0,
    receiver_absmax=float(receiver_a.abs().max()),
)
deterministic_error = vu.max_abs_difference(receiver_a, receiver_b)
vu.record_check(
    CHECKS,
    "deterministic repeated CPU forward run",
    deterministic_error == 0.0,
    max_abs_difference=deterministic_error,
)


In [ ]:
def expect_exception(name, exception_type, callable_object):
    caught = None
    try:
        callable_object()
    except Exception as exc:
        caught = exc
    vu.record_check(
        CHECKS,
        name,
        isinstance(caught, exception_type),
        expected=exception_type.__name__,
        received=None if caught is None else type(caught).__name__,
        message=None if caught is None else str(caught),
    )

common_arguments = dict(
    device=DEVICE,
    dx=dx,
    dt=dt,
    source_amplitudes=source,
    source_location=source_location,
    receiver_location=receiver_location,
    er=er,
    se=se,
    pmlthick=pml,
)
expect_exception(
    "relative permittivity below one is rejected",
    ValueError,
    lambda: DeepGPR.compute(**{**common_arguments, "er": torch.full_like(er, 0.9)}),
)
expect_exception(
    "unstable CFL time step is rejected",
    ValueError,
    lambda: DeepGPR.compute(**{**common_arguments, "dt": 1.0e-8}),
)
expect_exception(
    "unsupported FDTD order is rejected",
    ValueError,
    lambda: DeepGPR.compute(**{**common_arguments, "fdtd_order": 6}),
)
expect_exception(
    "overlapping CPML leaves no physical interior",
    ValueError,
    lambda: DeepGPR.compute(**{**common_arguments, "pmlthick": 10}),
)

pml_source = torch.tensor([[[4, 8, 0]]], dtype=torch.int32)
with warnings.catch_warnings(record=True) as captured:
    warnings.simplefilter("always")
    DeepGPR.compute(
        **{**common_arguments, "source_location": pml_source}
    )
warning_messages = [str(item.message) for item in captured]
vu.record_check(
    CHECKS,
    "acquisition point inside CPML emits a warning",
    any("inside CPML" in message for message in warning_messages),
    warnings=warning_messages,
)


In [ ]:
REPORT_PATH = vu.save_report(
    "00_local_backend_and_contracts",
    CHECKS,
    METADATA,
)
print(f"Completed {len(CHECKS)} required checks.")
